In [ ]:
import sys
import os
import json
from pathlib import Path
import importlib

# 1. SETUP DE RUTAS
current_path = Path(os.getcwd()) # .../actions/hidden_notebooks
actions_path = current_path.parent # .../actions/
base_path = actions_path.parent    # .../subtask01_proc_single/

if str(base_path) not in sys.path:
    sys.path.insert(0, str(base_path))

# 2. CARGA DE MÓDULOS
from actions import action01_gen_plan_proc_single
importlib.reload(action01_gen_plan_proc_single)

# 3. PARÁMETROS PARA LA PRUEBA
# Usamos el día 3 del 2026 que es donde tienes el archivo .nc según tu 'tree'
params = {
    "year": 2026,
    "day": 3,
    "sat_pos": "19",
    "product_id": "ABI-L2-MCMIPF",
    "output_folder_base": str(current_path / "f02_processed_test"),
    "dict_output_names": {"test_img": "{product}_test.png"}, # Schema dummy para la prueba
    "fnp_tag": "fnp01",
    "overwrite": True
}

print(f"🚀 Ejecutando Action01 para generar el plan...")

try:
    path_plan = action01_gen_plan_proc_single.gen_and_save_plan_proc_single(**params)
    print(f"✅ Plan generado en: {path_plan}")
    
    # 4. INSPECCIÓN INMEDIATA DEL CONTENIDO
    with open(path_plan, 'r') as f:
        plan_data = json.load(f)
    
    inventory = plan_data.get("proc_single_inventory", {})
    print(f"\n📊 Resumen del Inventario:")
    print(f"   - Escenas encontradas: {len(inventory)}")
    
    if len(inventory) > 0:
        first_key = list(inventory.keys())[0]
        first_item = inventory[first_key]
        path_abs = first_item["input_ref"].get("path_absolute")
        
        print(f"   - Primera escena: {first_key}")
        print(f"   - Path Absolute: {path_abs}")
        
        if path_abs is None:
            print("\n❌ ALERTA: El path_absolute es None.")
            print("Esto significa que Action01 encuentra el archivo pero no puede resolver su ruta completa.")
            print("Revisa la función de búsqueda en action01_gen_plan_proc_single.py")
        else:
            print(f"   - ¿Existe el archivo en disco?: {Path(path_abs).exists()}")
    else:
        print("\n❌ ALERTA: El inventario está VACÍO.")
        print("Action01 no encontró ningún archivo .nc para esos parámetros.")

except Exception as e:
    print(f"💥 Error al ejecutar Action01: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
import json
from actions.fn01_file_name_plan_proc_single import get_plan_proc_single_file_path

# 1. Obtenemos la ruta del plan que se acaba de crear
path_plan = get_plan_proc_single_file_path(
    year="2026", day="3", sat_id="19", 
    product_id="ABI-L2-MCMIPF", proc_tag="fnp01"
)

print(f"📄 Analizando Plan: {path_plan}")

if path_plan.exists():
    with open(path_plan, 'r') as f:
        plan = json.load(f)
    
    # 2. Extraemos la primera escena
    inventory = plan.get("proc_single_inventory", {})
    if inventory:
        first_fid = list(inventory.keys())[0]
        item = inventory[first_fid]
        
        expected_nc = item["input_ref"]["path_absolute"]
        is_ready = item["status"]["is_ready_to_proc"]
        error_msg = item["status"].get("error", "Sin error registrado")

        print(f"\n🔍 [DATOS DE LA ESCENA {first_fid}]")
        print(f"✅ ¿Está lista para procesar?: {is_ready}")
        print(f"❌ Error en Action02: {error_msg}")
        print(f"📂 Ruta que busca el código: \n   {expected_nc}")
        
        # 3. Verificación física
        exists = Path(expected_nc).exists()
        print(f"\n❓ ¿El archivo existe REALMENTE en esa ruta?: {'SI' if exists else 'NO'}")
    else:
        print("⚠️ El inventario del plan está vacío.")
else:
    print("❌ No se encontró el archivo del plan.")